# Module B · N4 — Freeze and the one-shot holdout> **This notebook does not freeze anything and does not open the holdout.**>> The freeze and the holdout run are gated command-line scripts on purpose. A> notebook cell is too easy to run twice, and the holdout is a one-shot: once an> evaluation has been seen, a second run is no longer blind.What this notebook does is rehearse both paths on a stand-in, so a bug is foundbefore the only blind run is spent.

In [ ]:
import sys, pathlibROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()sys.path.insert(0, str(ROOT))import numpy as np, pandas as pdpd.set_option("display.width", 200); pd.set_option("display.max_columns", 80)from moduleb import (baselines, config, contract, cv, dataio, envelope,                     freeze, guards, metrics, models, reason_codes)from moduleb.constants import PARAMS, TARGET_COLSfrom moduleb.features import add_features, make_foldsprint("moduleb ready — frozen config digest", config.frozen_config_digest()[:16])

## 1. The rehearsal`scripts/10_dryrun_freeze_predict.py` freezes on train lots only, then predictson the calibration file with its `168h` columns stripped — structurally identicalto the holdout, same guard path, no answers. Because the real answers exist, therehearsal can also be scored, which previews what the holdout run will looklike.

In [ ]:
p = dataio.RESULTS / "dryrun" / "dryrun_predictions.csv"display(pd.read_csv(p).head() if p.exists()        else "run: python scripts/10_dryrun_freeze_predict.py")

## 2. The output contract Anushka receivesRead from `data/REFERENCE/ModuleB_Output_Contract.csv`, not from code — if thereference file and the code ever disagree, the file wins.

In [ ]:
ref = dataio.expected_contract_columns()for c in ref: print(" ", c)print()print("module_b_disposition emitted:", "module_b_disposition" in ref,      "— disposition belongs to Module A + fusion (decision D1)")

## 3. The gates, and what each one refuses**Freeze** — `python scripts/11_freeze.py --team-signoff "<who, when>"`* refuses if the unit tests fail* refuses if the pre-declared fall-time rule has not been executed* refuses if `moduleb.config` disagrees with the executed decision* refuses to overwrite an existing frozen artifact without `--force --reason`**Holdout** — `python scripts/12_predict_holdout.py --frozen`* refuses if there is no frozen artifact* refuses if the config digest has changed since the freeze (stale artifact)* refuses if the holdout file carries `168h` columns (that is a data bug)* refuses if the output file already exists — the one-shot is spent

In [ ]:
from pathlib import Pathmodel = ROOT / "models" / "module_b_final01.joblib"out   = ROOT / "results" / "ModuleB_Final_Holdout_Predictions.csv"print("frozen artifact exists :", model.exists())print("holdout run spent      :", out.exists())

## 4. After the run* **Sanskruti** — `ModuleB_Final_Holdout_Predictions.csv`, for blind evaluation  against the hidden 168 h targets. Do not ask for the targets.* **Anushka** — the same file, plus `models/module_b_final01.joblib` and  `docs/INTEGRATION_NOTE.md`.A worse holdout score than calibration is an ordinary result, not a bug, and nota reason to change anything.